# Chapter 11: Unstructured Data and Entity Linking

**Level 3: Integration & Engineering** · ⏱ about 3 hours · Dataset: *EduGraph* + lesson documents

### What you will learn

1. How documents become knowledge: **extract → link → store with provenance → use**
2. **Concept extraction** with a local LLM (Ollama, `qwen2.5:7b`) in JSON mode
3. **Entity linking**: exact matching, **embeddings** (`nomic-embed-text`) and LLM choice, and how to **measure** them
4. Why extracted facts need **confidence, status and provenance**, and how to model them (PROV-O, one named graph per document)
5. **Using** the result: reading lists for weak students, curriculum checks, and a human review queue

### Before you start

Ollama must be running locally with two models:

```
ollama serve
ollama pull qwen2.5:7b
ollama pull nomic-embed-text
```

LLM output varies from run to run, so extractions are **cached** in `data/edugraph/lessons/extractions.json`, keyed by document text, model and prompt version. The notebook reuses them. Delete the file to extract afresh; your results will then differ slightly from the text.

In [1]:
import sys
from pathlib import Path

SRC = str(Path('..', 'src').resolve())
if SRC not in sys.path:
    sys.path.insert(0, SRC)

from datetime import datetime, timezone

import pandas as pd
import stardog
from rdflib import BNode, Graph, Literal, Namespace, URIRef
from rdflib.namespace import PROV, RDF, RDFS, XSD

from kg import client, extraction, imports, sparql

pd.set_option('display.max_colwidth', 70)
pd.set_option('display.width', 200)
conn = client.connect()
PREFIX = 'urn:tutorial:ch11'
ROOT = Path('..')
EDU_DIR = ROOT / 'data' / 'edugraph'
LESSONS = EDU_DIR / 'lessons'
CACHE = LESSONS / 'extractions.json'

EDU = Namespace('http://example.org/edu#')
DOC = Namespace('http://example.org/edu/document/')
LINK = Namespace('http://example.org/edu/link/')
CONCEPT = Namespace('http://example.org/edu/concept/')

print('Ollama ready with both models:', extraction.ollama_available())

Ollama ready with both models: True


## 11.1 From documents to knowledge

Much of what an organization knows sits in **documents**: lesson notes, reports, contracts, e-mails. A knowledge graph becomes far more useful when it knows what those documents are **about**, and links them to the things it already models.

```
 lessons/*.md ──► 1. EXTRACT ──► 2. LINK ─────────────► 3. STORE ────────────────► 4. USE
                  LLM finds      match each mention      one named graph per        reading lists,
                  concepts by    to a curriculum topic   document + PROV-O:         curriculum checks,
                  role           (exact / embedding /    who extracted what, when,  review queue
                                 LLM), with a score      with which confidence
```

**Stardog's own pipeline, BITES**, stores documents in Stardog and extracts entities from them on the server. It isn't available on this tutorial's Stardog Cloud endpoint (its API returns `404`), so we build the pipeline in Python with a local LLM. The ideas (mentions, linking, confidence, provenance) are the same in any tool.

> 🔁 **Neo4j equivalent:** Neo4j's GenAI and LLM Graph Builder tools do the same extract-and-link job into a property graph. The design questions in this chapter (thresholds, review, provenance) apply there unchanged.

## 11.2 Setup: the graph we link to

The curriculum (Chapter 10) is the **target vocabulary**: every mention should end up linked to one of its 21 topics, or be flagged as new. The SIS data is loaded too, for the reading lists in 11.7.

In [2]:
CURRICULUM = f'{PREFIX}:curriculum'
SIS = {name: f'{PREFIX}:sis:{name}' for name in ('students', 'enrollments', 'assessments')}

with client.transaction(conn):
    conn.clear(graph_uri=CURRICULUM)
    conn.add(stardog.content.File(str(EDU_DIR / 'curriculum.ttl')), graph_uri=CURRICULUM)
for name, graph in SIS.items():
    imports.import_file(EDU_DIR / 'sis' / f'{name}.csv', ROOT / 'ontology' / 'mappings' / f'{name}.sms', graph)

topics = dict(sparql.run_query(conn, 'PREFIX edu: <http://example.org/edu#> SELECT ?t ?name { ?t a edu:Topic ; edu:name ?name }',
                               graphs=CURRICULUM).values)
print(len(topics), 'topics:', ', '.join(sorted(topics.values())))

21 topics: Algebra refresher, Calculus basics, Classification, Clustering, Control flow, Data structures, Data visualization, Functions, Knowledge graphs, Linear algebra, Linear regression, Model evaluation, Neural networks, NumPy, Probability, Python basics, RDF and SPARQL, SQL basics, SQL joins, Statistics, pandas


## 11.3 The documents

Six short lesson texts, as a teacher might write them:

In [3]:
docs = {f.stem: f.read_text(encoding='utf-8') for f in sorted(LESSONS.glob('*.md'))}
for name, text in docs.items():
    print(f'{name:26} {len(text.split()):4} words  "{text.splitlines()[0][2:]}"')
print()
print(docs['linear-regression'])

evaluating-classifiers      101 words  "Evaluating classifiers"
linear-regression           127 words  "Understanding linear regression"
pandas-dataframes           142 words  "Getting started with pandas DataFrames"
plotting-with-matplotlib     99 words  "Plotting with matplotlib"
probability-basics           96 words  "Probability for data scientists"
sql-to-sparql               120 words  "From SQL joins to SPARQL"

# Understanding linear regression

Linear regression predicts a number, such as a house price, from one or more input features by fitting a straight line (or a plane) through the data. To follow this lesson you need basic statistics, in particular mean, variance and correlation, and some linear algebra: vectors, matrices and the matrix product.

We fit the line by minimising the mean squared error. For small problems there is a closed-form solution using matrix inversion; for large ones we use gradient descent, which repeatedly nudges the coefficients downhill on the err

## 11.4 Step 1: extract concepts with an LLM

We ask the LLM for concepts in three **roles**, because the role matters as much as the concept:

- **teaches**: what the lesson is about
- **requires**: what a reader must know first
- **mentions**: other named things that come up

Three settings make LLM output usable in a pipeline:

- **JSON mode** (`format: json`): the model must return valid JSON.
- **Temperature 0 and a fixed seed**: as repeatable as the model allows.
- **A strict prompt**: limits ("1 to 3", "up to 5"), and examples of what *not* to return (function names, code).

In [4]:
print(extraction.PROMPT.replace('<<TEXT>>', '…the lesson text…'))

You extract learning concepts from a lesson, for a curriculum knowledge graph.
Return only JSON of the form {"teaches": [...], "requires": [...], "mentions": [...]}.

- "teaches": the 1 to 3 main subjects the lesson teaches. Broad subject names such as
  "linear regression" or "pandas", never function names, code or single steps.
- "requires": up to 5 subjects the reader must already know before this lesson.
- "mentions": up to 6 other named methods, tools or concepts that appear in the lesson but are
  neither its main subject nor a prerequisite.
Use short names of 1 to 4 words, in lower case, with no explanations.

Lesson:
"""
…the lesson text…
"""


In [5]:
extracted = {name: extraction.extract(text, cache=CACHE) for name, text in docs.items()}
rows = [(doc, role, ', '.join(ms)) for doc, ex in extracted.items() for role, ms in ex.items()]
pd.DataFrame(rows, columns=['document', 'role', 'concepts']).pivot(index='document', columns='role', values='concepts')[['teaches', 'requires', 'mentions']]

role,teaches,requires,mentions
document,,,
evaluating-classifiers,"classification evaluation, confusion matrix, precision recall f1 s...","logistic regression, decision trees, confusion matrix, cross valid...","overfitting, learning curves"
linear-regression,linear regression,"statistics, linear algebra, pandas","mean, variance, correlation, matrix inversion, gradient descent, r..."
pandas-dataframes,"pandas, dataframes","python lists, python dictionaries, numpy arrays","pd.read_csv, head(), groupby, mean, count, isna(), fillna(), dropn..."
plotting-with-matplotlib,"plotting with matplotlib, data visualization","pandas dataframes, data filtering","line charts, bar charts, histograms, scatter plots, subplots, colo..."
probability-basics,"probability, distributions, statistics","fractions, basic algebra, independent events, conditional probability","bayes' theorem, random variables, expected value, variance, confid..."
sql-to-sparql,"sparql, sql joins, knowledge graphs","sql, select, where, joins","rdf triples, iri, prefixes, property paths, recursive query"


Look closely before trusting it:

- Mostly sensible: the regression lesson teaches *linear regression* and requires *statistics*, *linear algebra* and *pandas*.
- **Role mistakes**: the SQL-to-SPARQL lesson "teaches" *sql joins*, but the text says the reader must *already know* them. The probability lesson "teaches" *statistics*, though it only says statistics builds on probability.
- **Granularity drift**: the pandas lesson still lists functions (`head()`, `fillna()`) under *mentions*.

LLM extraction is **fast and broad, but not reliable**. Everything downstream must treat it as a *suggestion* with a confidence, not as a fact.

## 11.5 Step 2: link mentions to topics

A mention like *"numpy arrays"* is just text. **Entity linking** decides which **resource** in the graph it refers to: `topic:numpy`. Three techniques, from strict to generous.

### Exact names

After normalizing case and punctuation, *"linear algebra"* matches the topic **Linear algebra** exactly. This is precise, but misses every synonym.

### Embeddings

An **embedding model** turns text into a vector (768 numbers for `nomic-embed-text`) such that texts with similar **meaning** have vectors pointing in similar directions. **Cosine similarity** (−1 … 1) measures that closeness:

In [6]:
names = sorted(topics.values())
topic_vectors = dict(zip(names, extraction.embed(names, kind='document')))
probe = ['numpy arrays', 'sql', 'python lists', 'logistic regression', 'house prices']
probe_vectors = dict(zip(probe, extraction.embed(probe)))

similarity = pd.DataFrame({m: {t: extraction.cosine(v, topic_vectors[t]) for t in names} for m, v in probe_vectors.items()})
top = {m: similarity[m].sort_values(ascending=False).head(3) for m in probe}
pd.DataFrame({m: [f'{t} ({s:.2f})' for t, s in top[m].items()] for m in probe}, index=['best', '2nd', '3rd'])

,numpy arrays,sql,python lists,logistic regression,house prices
best,NumPy (0.74),SQL basics (0.79),Python basics (0.63),Linear regression (0.71),RDF and SPARQL (0.44)
2nd,Data structures (0.49),SQL joins (0.78),NumPy (0.52),Clustering (0.52),NumPy (0.41)
3rd,Algebra refresher (0.48),RDF and SPARQL (0.63),RDF and SPARQL (0.52),Statistics (0.52),Linear regression (0.40)


- *numpy arrays → NumPy* and *sql → SQL basics*: right, with a clear margin.
- *python lists → Python basics*: close, but our curriculum puts lists under **Data structures**.
- *logistic regression → Linear regression*: wrong. The names look alike, but it's a **classification** method. Embeddings see words, not meaning in context.
- *house prices*: rightly matches nothing well.

Embeddings need a **threshold**: accept the best match only if it's similar enough.

### LLM choice

For mentions below the threshold, ask the LLM to pick **from the list of topics**, or answer `null`. It knows that logistic regression is about classification. The answer is only accepted if it's **exactly** one of the listed names:

In [7]:
print(extraction.CHOOSE_PROMPT.replace('<<TOPICS>>', '…21 topic names…'))
for m in ['logistic regression', 'python lists', 'house prices']:
    print(f'{m:22} -> {extraction.choose_topic(m, names, cache=CACHE)}')

In a data science curriculum, a lesson refers to the concept "<<MENTION>>".
Which ONE of these curriculum topics is that concept part of? Topics: …21 topic names….
Answer with JSON {"topic": "<exact topic name>"}, or {"topic": null} if none of them clearly fits.
logistic regression    -> Classification
python lists           -> Python basics
house prices           -> Linear regression


### Measure, don't guess

Which method, which threshold? **Measure** against a small **gold standard**: mentions labelled by hand with the correct topic, or `None` when no topic fits.

In [8]:
GOLD = {
    'logistic regression': 'Classification', 'decision trees': 'Classification', 'confusion matrix': 'Model evaluation',
    'cross validation': 'Model evaluation', 'statistics': 'Statistics', 'linear algebra': 'Linear algebra', 'pandas': 'pandas',
    'python lists': 'Data structures', 'python dictionaries': 'Data structures', 'numpy arrays': 'NumPy',
    'pandas dataframes': 'pandas', 'data filtering': 'pandas', 'fractions': 'Algebra refresher', 'basic algebra': 'Algebra refresher',
    'independent events': 'Probability', 'conditional probability': 'Probability', 'sql': 'SQL basics', 'select': 'SQL basics',
    'where': 'SQL basics', 'joins': 'SQL joins', 'bayes theorem': 'Probability', 'matplotlib': 'Data visualization',
    'gradient descent': None, 'color theory': None, 'house prices': None,
}
scored = extraction.Linker(topics, accept_threshold=2, use_llm=False).link(list(GOLD))      # nothing accepted: just scores
llm = {m: extraction.choose_topic(m, names, cache=CACHE) for m in GOLD}

def evaluate(predictions):
    correct = sum(1 for m, p in predictions.items() if p is not None and p == GOLD[m])
    wrong = sum(1 for m, p in predictions.items() if p is not None and p != GOLD[m])
    missed = sum(1 for m, p in predictions.items() if GOLD[m] is not None and p != GOLD[m] and p is None)
    return {'correct': correct, 'wrong': wrong, 'missed': missed}

def by_embedding(s, t):
    '''Exact name match, else the most similar topic if it scores at least t.'''
    return s['topic'] if s['method'] == 'exact' else (s['best_guess'] if s['score'] >= t else None)

results = {}
for t in (0.60, 0.70, 0.75, 0.80):
    results[f'embedding ≥ {t}'] = evaluate({s['mention']: by_embedding(s, t) for s in scored})
results['LLM choice only'] = evaluate(llm)
results['hybrid: embedding ≥ 0.75, else LLM'] = evaluate(
    {s['mention']: by_embedding(s, 0.75) or llm[s['mention']] for s in scored})
pd.DataFrame(results).T

,correct,wrong,missed
embedding ≥ 0.6,8,4,10
embedding ≥ 0.7,6,1,15
embedding ≥ 0.75,5,0,17
embedding ≥ 0.8,3,0,19
LLM choice only,16,9,0
"hybrid: embedding ≥ 0.75, else LLM",17,8,0


The classic trade-off:

- **Strict embeddings** (≥ 0.75) are almost never wrong, but miss most links.
- **The LLM** finds many more, but also **invents** links for things that fit nothing (e.g. *house prices*).
- **Hybrid** gets the most right.

So we don't treat all links alike. The linker gives each one a **status**:

| Method | Status | Meaning |
|---|---|---|
| exact name, or embedding ≥ 0.75 | **accepted** | trusted automatically |
| LLM choice | **review** | a person confirms or rejects it |
| nothing fits | **new** | a candidate for a new concept |

## 11.6 Step 3: store with provenance

Extracted facts are **claims made by a model**, and the graph must say so. Three modelling choices:

1. **Link resources**, not bare triples. Each link records the mention, role, method, score and status, so it can be reviewed, filtered and explained:
   ```turtle
   link:linear-regression-requires-statistics a edu:TopicLink ;
       edu:document doc:linear-regression ; edu:role "requires" ; edu:topic topic:statistics ;
       edu:mention "statistics" ; edu:method "exact" ; edu:score 1.0 ; edu:status "accepted" .
   ```
   **Accepted** links also get a direct shortcut, e.g. `doc:linear-regression edu:requiresKnowledgeOf topic:statistics`, so queries stay simple.
2. **One named graph per document**: everything extracted from a lesson can be replaced or removed together.
3. **PROV-O**, the W3C provenance vocabulary, in a separate graph: which **activity** (an extraction run) produced each graph, from which **document**, with which **agent** (model and prompt version), and when.

In [9]:
ROLE_PROPERTY = {'teaches': EDU.teaches, 'requires': EDU.requiresKnowledgeOf, 'mentions': EDU.mentions}
linker = extraction.Linker(topics, accept_threshold=0.75, use_llm=True, cache=CACHE)
AGENT = URIRef(f'http://example.org/edu/agent/{extraction.EXTRACT_MODEL.replace(":", "-")}-prompt-{extraction.PROMPT_VERSION}')
PROVENANCE = f'{PREFIX}:provenance'

def slug(text):
    return extraction.normalize(text).replace(' ', '-')

def document_graph(name, text, extracted):
    '''All triples extracted from one document, plus its links.'''
    g = Graph()
    d = DOC[name]
    g.add((d, RDF.type, EDU.Document))
    g.add((d, EDU.title, Literal(text.splitlines()[0].lstrip('# '))))
    g.add((d, EDU.source, Literal(f'data/edugraph/lessons/{name}.md')))
    all_links = []
    for role, mentions in extracted.items():
        for l in linker.link(mentions):
            all_links.append({'document': name, 'role': role, **l})
            if l['status'] == 'new':
                c = CONCEPT[slug(l['mention'])]
                g.add((c, RDF.type, EDU.CandidateConcept))
                g.add((c, EDU.name, Literal(l['mention'])))
                g.add((c, EDU.mentionedIn, d))
                continue
            link = LINK[f'{name}-{role}-{slug(l["mention"])}']
            for p, o in [(RDF.type, EDU.TopicLink), (EDU.document, d), (EDU.role, Literal(role)),
                         (EDU.topic, URIRef(l['iri'])), (EDU.mention, Literal(l['mention'])),
                         (EDU.method, Literal(l['method'])), (EDU.score, Literal(l['score'], datatype=XSD.decimal)),
                         (EDU.status, Literal(l['status']))]:
                g.add((link, p, o))
            if l['status'] == 'accepted':
                g.add((d, ROLE_PROPERTY[role], URIRef(l['iri'])))
    return g, all_links

def provenance(name, graph_iri, started, ended):
    g = Graph()
    run = URIRef(f'http://example.org/edu/run/{name}-{started:%Y%m%dT%H%M%S}')
    g.add((URIRef(graph_iri), RDF.type, PROV.Entity))
    g.add((URIRef(graph_iri), PROV.wasDerivedFrom, DOC[name]))
    g.add((URIRef(graph_iri), PROV.wasGeneratedBy, run))
    g.add((run, RDF.type, PROV.Activity))
    g.add((run, PROV.used, DOC[name]))
    g.add((run, PROV.wasAssociatedWith, AGENT))
    g.add((run, PROV.startedAtTime, Literal(started, datatype=XSD.dateTime)))
    g.add((run, PROV.endedAtTime, Literal(ended, datatype=XSD.dateTime)))
    g.add((AGENT, RDF.type, PROV.SoftwareAgent))
    g.add((AGENT, RDFS.label, Literal(f'{extraction.EXTRACT_MODEL} + {extraction.EMBED_MODEL}, prompt {extraction.PROMPT_VERSION}')))
    return g

links = []
with client.transaction(conn):
    conn.clear(graph_uri=PROVENANCE)
for name, text in docs.items():
    graph_iri = f'{PREFIX}:extracted:{name}'
    started = datetime.now(timezone.utc)
    g, doc_links = document_graph(name, text, extracted[name])
    ended = datetime.now(timezone.utc)
    links += doc_links
    with client.transaction(conn):
        conn.clear(graph_uri=graph_iri)
        conn.add(stardog.content.Raw(g.serialize(format='turtle').encode(), 'text/turtle'), graph_uri=graph_iri)
        conn.add(stardog.content.Raw(provenance(name, graph_iri, started, ended).serialize(format='turtle').encode(),
                                     'text/turtle'), graph_uri=PROVENANCE)
    print(f'{name:26} {len(g):4} triples -> {graph_iri}')

links = pd.DataFrame(links)
links.groupby(['status', 'method']).size().rename('links').reset_index()

evaluating-classifiers       91 triples -> urn:tutorial:ch11:extracted:evaluating-classifiers


linear-regression            82 triples -> urn:tutorial:ch11:extracted:linear-regression


pandas-dataframes           117 triples -> urn:tutorial:ch11:extracted:pandas-dataframes


plotting-with-matplotlib     84 triples -> urn:tutorial:ch11:extracted:plotting-with-matplotlib


probability-basics          109 triples -> urn:tutorial:ch11:extracted:probability-basics


sql-to-sparql                94 triples -> urn:tutorial:ch11:extracted:sql-to-sparql


,status,method,links
0,accepted,embedding,4
1,accepted,exact,10
2,new,none,3
3,review,llm,53


The accepted links, the ones queries will trust:

In [10]:
links[links.status == 'accepted'][['document', 'role', 'mention', 'topic', 'method', 'score']].sort_values(['document', 'role'])

,document,role,mention,topic,method,score
12,linear-regression,requires,statistics,Statistics,exact,1.000
13,linear-regression,requires,linear algebra,Linear algebra,exact,1.000
14,linear-regression,requires,pandas,pandas,exact,1.000
11,linear-regression,teaches,linear regression,Linear regression,exact,1.000
34,pandas-dataframes,mentions,sql join,SQL joins,embedding,0.848
21,pandas-dataframes,teaches,pandas,pandas,exact,1.000
36,plotting-with-matplotlib,teaches,data visualization,Data visualization,exact,1.000
45,probability-basics,teaches,probability,Probability,exact,1.000
47,probability-basics,teaches,statistics,Statistics,exact,1.000
61,sql-to-sparql,requires,sql,SQL basics,embedding,0.793


## 11.7 Step 4: use the knowledge

All graphs of this chapter, for the queries below:

In [11]:
EXTRACTED = [f'{PREFIX}:extracted:{name}' for name in docs]
ALL = [CURRICULUM, *SIS.values(), *EXTRACTED, PROVENANCE]
PREFIXES = '''PREFIX edu: <http://example.org/edu#>
PREFIX prov: <http://www.w3.org/ns/prov#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
'''
def q(query, **params):
    return sparql.run_query(conn, PREFIXES + query, graphs=ALL, **params)

### A reading list for weak prerequisites

Chapter 10 found students whose weak topics block others. Now the graph also knows which **documents teach** which topic, so it can recommend reading:

In [12]:
gaps = sparql.run_query(conn, 'edugraph/prerequisite_gaps', graphs=ALL, threshold=50)
teaching = q('''SELECT ?weak_topic ?document ?title
WHERE { ?d edu:teaches ?t ; edu:title ?title . ?t edu:name ?weak_topic  BIND(STRAFTER(STR(?d), "/document/") AS ?document) }''')
reading = gaps.merge(teaching, on='weak_topic')[['student_name', 'weak_topic', 'score', 'title']].drop_duplicates()
print(f'{reading.student_name.nunique()} of {gaps.student_name.nunique()} students with gaps get at least one recommendation')
reading.sort_values(['student_name', 'score']).head(10)

15 of 18 students with gaps get at least one recommendation


,student_name,weak_topic,score,title
0,Aarav Sharma,Statistics,33,Probability for data scientists
1,Ananya Singh,Statistics,39,Probability for data scientists
2,Ananya Singh,RDF and SPARQL,45,From SQL joins to SPARQL
3,Chen Tanaka,Probability,21,Probability for data scientists
4,Dara Mueller,pandas,49,Getting started with pandas DataFrames
5,Dara Mueller,Probability,49,Probability for data scientists
7,Elena Rossi,pandas,44,Getting started with pandas DataFrames
8,Emil Rossi,pandas,38,Getting started with pandas DataFrames
9,Isla Garcia,pandas,18,Getting started with pandas DataFrames
11,Ivan Singh,pandas,36,Getting started with pandas DataFrames


Look at *Statistics*: the recommended document is the **probability** lesson, because the extractor claimed it *teaches* statistics (11.4). The link was even "accepted", because the **name** matched exactly. The match said nothing about whether the **role** was right. Extraction errors flow straight into recommendations, which is why the review queue below matters, and why a production system would also review roles.

### Curriculum checks

When a document **requires** a topic for something it **teaches**, the curriculum should list that as a prerequisite too. Where it doesn't, the curriculum may be missing an edge, or the extraction is wrong. Either way, a person should look:

In [13]:
q('''SELECT ?document ?teaches ?requires
WHERE {
  ?d edu:teaches ?t ; edu:requiresKnowledgeOf ?r .
  FILTER(?t != ?r)
  FILTER NOT EXISTS { ?t edu:requires+ ?r }                 # the curriculum doesn't know this dependency
  ?t edu:name ?teaches . ?r edu:name ?requires
  BIND(STRAFTER(STR(?d), "/document/") AS ?document)
}
ORDER BY ?document''')

,document,teaches,requires


**No rows**: every dependency among the *accepted* links is already in the curriculum, so the check confirms the curriculum. As reviewers accept more links, more pairs become testable, and this query becomes a standing check that runs after every extraction.

### The review queue

The links the LLM suggested are waiting for a person. A simple queue, lowest similarity first:

In [14]:
queue = q('''SELECT ?link ?document ?role ?mention ?suggested ?score
WHERE { ?link a edu:TopicLink ; edu:status "review" ; edu:document ?d ; edu:role ?role ; edu:mention ?mention ;
              edu:topic/edu:name ?suggested ; edu:score ?score
        BIND(STRAFTER(STR(?d), "/document/") AS ?document) }
ORDER BY ?score''')
print(len(queue), 'links to review')
queue.head(8)

53 links to review


,link,document,role,mention,suggested,score
0,http://example.org/edu/link/evaluating-classifiers-mentions-overfi...,evaluating-classifiers,mentions,overfitting,Model evaluation,0.395
1,http://example.org/edu/link/probability-basics-requires-independen...,probability-basics,requires,independent events,Probability,0.400
2,http://example.org/edu/link/evaluating-classifiers-teaches-precisi...,evaluating-classifiers,teaches,precision recall f1 score,Model evaluation,0.418
3,http://example.org/edu/link/plotting-with-matplotlib-teaches-plott...,plotting-with-matplotlib,teaches,plotting with matplotlib,Data visualization,0.429
4,http://example.org/edu/link/evaluating-classifiers-teaches-roc-curve,evaluating-classifiers,teaches,roc curve,Model evaluation,0.450
5,http://example.org/edu/link/pandas-dataframes-mentions-dropna,pandas-dataframes,mentions,dropna(),pandas,0.452
6,http://example.org/edu/link/sql-to-sparql-mentions-property-paths,sql-to-sparql,mentions,property paths,Knowledge graphs,0.453
7,http://example.org/edu/link/pandas-dataframes-mentions-pd-read-csv,pandas-dataframes,mentions,pd.read_csv,pandas,0.453


A reviewer's decisions are just updates. Accepting sets the status and adds the shortcut triple; rejecting only changes the status, and the link stays as a record that someone said no. Two decisions: *logistic regression → Classification* is right; *property paths → Knowledge graphs* is wrong (property paths belong to RDF and SPARQL):

In [15]:
def review(link_iri, accept: bool):
    decision = 'accepted' if accept else 'rejected'
    conn.update(PREFIXES + f'''
    DELETE {{ GRAPH ?g {{ ?link edu:status "review" }} }}
    INSERT {{ GRAPH ?g {{ ?link edu:status "{decision}" ; edu:reviewedAt ?now .
                          {'?d ?shortcut ?t .' if accept else ''} }} }}
    WHERE  {{ GRAPH ?g {{ ?link edu:status "review" ; edu:document ?d ; edu:topic ?t ; edu:role ?role }}
              VALUES (?role ?shortcut) {{ ("teaches" edu:teaches) ("requires" edu:requiresKnowledgeOf) ("mentions" edu:mentions) }}
              BIND(NOW() AS ?now) }}''', bindings={'link': sparql.term(URIRef(link_iri))})

def find(mention, document):
    return queue[(queue.mention == mention) & (queue.document == document)].link.iloc[0]

review(find('logistic regression', 'evaluating-classifiers'), accept=True)     # LLM said Classification: correct
review(find('property paths', 'sql-to-sparql'), accept=False)                     # LLM said Knowledge graphs: really RDF and SPARQL

q('''SELECT ?mention ?topic ?status ?reviewedAt
WHERE { ?l edu:status ?status ; edu:mention ?mention ; edu:topic/edu:name ?topic ; edu:reviewedAt ?reviewedAt }''')

,mention,topic,status,reviewedAt
0,property paths,Knowledge graphs,rejected,2026-09-27 11:55:16.540000+00:00
1,logistic regression,Classification,accepted,2026-09-27 11:55:16.095000+00:00


### Provenance: who said what?

Every extracted fact can be traced: which graph it's in, which run produced that graph, and which model and prompt that run used:

In [16]:
q('''SELECT ?document ?graph ?agent ?started (COUNT(?link) AS ?links)
WHERE {
  ?graph prov:wasDerivedFrom ?d ; prov:wasGeneratedBy ?run .
  ?run prov:wasAssociatedWith/rdfs:label ?agent ; prov:startedAtTime ?started .
  ?link a edu:TopicLink ; edu:document ?d .
  BIND(STRAFTER(STR(?d), "/document/") AS ?document)
}
GROUP BY ?document ?graph ?agent ?started ORDER BY ?document''')

,document,graph,agent,started,links
0,evaluating-classifiers,urn:tutorial:ch11:extracted:evaluating-classifiers,"qwen2.5:7b + nomic-embed-text, prompt v2",2026-09-27 11:54:26.523303+00:00,11
1,linear-regression,urn:tutorial:ch11:extracted:linear-regression,"qwen2.5:7b + nomic-embed-text, prompt v2",2026-09-27 11:54:34.559208+00:00,9
2,pandas-dataframes,urn:tutorial:ch11:extracted:pandas-dataframes,"qwen2.5:7b + nomic-embed-text, prompt v2",2026-09-27 11:54:42.534636+00:00,14
3,plotting-with-matplotlib,urn:tutorial:ch11:extracted:plotting-with-matplotlib,"qwen2.5:7b + nomic-embed-text, prompt v2",2026-09-27 11:54:50.592333+00:00,10
4,probability-basics,urn:tutorial:ch11:extracted:probability-basics,"qwen2.5:7b + nomic-embed-text, prompt v2",2026-09-27 11:54:58.576628+00:00,13
5,sql-to-sparql,urn:tutorial:ch11:extracted:sql-to-sparql,"qwen2.5:7b + nomic-embed-text, prompt v2",2026-09-27 11:55:06.805177+00:00,10


Because each document's extraction lives in **its own named graph**, fixing a bad run is easy. Re-extract with a new prompt version, then replace the graph, or drop every graph produced by the old agent. No other data is touched.

> **Why not attach confidence to each triple directly?** With RDF-star (edge properties, Chapter 1) you could annotate `<< doc:x edu:teaches topic:y >> edu:score 0.8`. It's off on this database, and link resources work everywhere. They also hold review decisions naturally.

## 11.8 Exercises

**Exercise 1.** List all **candidate concepts** (status `new`) with the documents that mention them. Which would you add to the curriculum?

<details><summary>Solution</summary>

```python
q('''SELECT ?concept (GROUP_CONCAT(?doc; separator=", ") AS ?documents)
WHERE { ?c a edu:CandidateConcept ; edu:name ?concept ; edu:mentionedIn ?d  BIND(STRAFTER(STR(?d), "/document/") AS ?doc) }
GROUP BY ?concept''')
```
</details>

**Exercise 2.** Rerun the evaluation in 11.5 with `accept_threshold` 0.70 for the hybrid. How do correct, wrong and missed change?

<details><summary>Solution</summary>

```python
evaluate({s['mention']: by_embedding(s, 0.70) or llm[s['mention']] for s in scored})
```
A lower threshold accepts more embedding matches without review, including *logistic regression → Linear regression* (0.71), a wrong one.
</details>

**Exercise 3.** Add a seventh lesson, e.g. `neural-networks.md` with a few sentences of your own. Extract, link and load it with the same functions (`extraction.extract`, `document_graph`, `provenance`), then check whether it shows up in reading lists.

<details><summary>Solution</summary>

```python
name = 'neural-networks'
text = '# Neural networks\n\nA neural network stacks layers of weighted sums and activation functions. You need classification and calculus (derivatives) first. We train with backpropagation and gradient descent.'
ex = extraction.extract(text, cache=CACHE)
g, new_links = document_graph(name, text, ex)
graph_iri = f'{PREFIX}:extracted:{name}'
with client.transaction(conn):
    conn.clear(graph_uri=graph_iri)
    conn.add(stardog.content.Raw(g.serialize(format='turtle').encode(), 'text/turtle'), graph_uri=graph_iri)
ALL.append(graph_iri)
pd.DataFrame(new_links)[['role', 'mention', 'topic', 'status']]
```
</details>

**Exercise 4.** Why is it risky to accept a link just because the **name** matches exactly? Find an example in this chapter.

<details><summary>Solution</summary>

A name match proves *which* topic is meant, not that the **role** is right. The probability lesson "teaches" *Statistics* (an exact name match, so accepted), but it doesn't teach statistics; it only says statistics builds on it. As a result, students weak in Statistics were sent to the probability lesson. Roles need review too, or a second check such as the curriculum check in 11.7.
</details>

In [17]:
# your answers here

## 11.9 Clean up

In [18]:
CLEAN_UP = True

if CLEAN_UP:
    graphs = sparql.run_query(conn, f'SELECT DISTINCT ?g {{ GRAPH ?g {{ ?s ?p ?o }} FILTER(STRSTARTS(STR(?g), "{PREFIX}")) }}').g.tolist()
    with client.transaction(conn):
        for g in graphs:
            conn.clear(graph_uri=g)
    print(f'Removed {len(graphs)} graphs')
conn.close()

Removed 11 graphs


## Summary

- Documents become knowledge through **extract → link → store with provenance → use**.
- A local **LLM in JSON mode** extracts concepts by **role** quickly, but makes role and granularity mistakes. Treat its output as suggestions.
- **Entity linking** maps mentions to graph resources. Exact names are precise but narrow; **embeddings** plus a threshold are precise at high thresholds; **LLM choice** from a fixed list has high recall but invents links. **Measure** against a gold standard, and combine them.
- Give every link a **method, score and status** (accepted / review / new). Automate the confident ones, and **route the rest to people**.
- Store each document's extraction in **its own named graph**, and describe runs with **PROV-O**: every fact can be traced, and a bad run can be replaced cleanly.
- Linked documents power **reading lists**, **curriculum checks** and **review queues**, and extraction errors flow into them, which is exactly why the checks matter.

**Next: Chapter 12, Search, similarity and geospatial.** Full-text search, "recipes similar to X", and distance queries, including which of these features can be switched on in Stardog Cloud.